# Python Data Cleaning

## Project Methodology — CRISP-DM

We are following the **CRISP-DM (Cross-Industry Standard Process for Data Mining)** methodology throughout this project.

The project follows a structured analytical workflow consisting of:

1. **Business Understanding** — Define business objectives, requirements, KPIs, and analytical goals.
2. **Data Understanding** — Explore the available data, understand its structure, assess data quality, and identify potential issues.
3. **Data Preparation** — Clean, transform, validate, and prepare the data for analysis.
4. **Modeling / Analysis** — Perform analytical techniques, exploratory analysis, and statistical analysis to generate insights.
5. **Evaluation** — Validate results against business requirements, KPIs, and data quality expectations.
6. **Deployment** — Deliver the final insights through dashboards, recommendations, and project documentation.

### Current CRISP-DM Phase

**Data Preparation → Data Cleaning**

This notebook focuses on preparing the datasets for reliable downstream analysis by identifying and addressing data quality issues while preserving the integrity of the original source data.

<img src="crisp-dm.png" width="500">

*Figure 1: CRISP-DM framework followed throughout the project.*

---

In [ ]:
# Step 1: Import required libraries

import pandas as pd
import mysql.connector

print("Libraries imported successfully!")

In [3]:
# Step 2: Connect to MySQL database

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="YOUR_MYSQL_PASSWORD",
    database="financial_performance"
)

print("MySQL connection successful!")

MySQL connection successful!


In [4]:
# Step 3: Load raw financial transactions

df_transactions = pd.read_sql(
    "SELECT * FROM financial_transactions",
    conn
)

print("Raw financial transactions loaded!")
print("Rows:", df_transactions.shape[0])
print("Columns:", df_transactions.shape[1])

C:\Users\Ayush Pandey\AppData\Local\Temp\ipykernel_24168\2532859087.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_transactions = pd.read_sql(


Raw financial transactions loaded!
Rows: 10400
Columns: 10


In [5]:
# Step 4: Missing value profiling

missing_values = pd.DataFrame({
    "Column": df_transactions.columns,
    "Missing_Count": df_transactions.isnull().sum().values,
    "Missing_Percentage": (
        df_transactions.isnull().mean().values * 100
    )
})

missing_values

,Column,Missing_Count,Missing_Percentage
0,transaction_id,0,0.000000
1,transaction_date,0,0.000000
2,amount,0,0.000000
3,account_type,0,0.000000
4,category,0,0.000000
5,business_unit,0,0.000000
6,region,0,0.000000
7,customer_id,4886,46.980769
8,vendor_id,6275,60.336538
9,description,0,0.000000


In [6]:
# Step 5: Validate customer/vendor ID completeness by account type

id_validation = df_transactions.groupby("account_type").agg(
    total_transactions=("transaction_id", "count"),
    missing_customer_id=("customer_id", lambda x: x.isna().sum()),
    missing_vendor_id=("vendor_id", lambda x: x.isna().sum())
)

id_validation

,total_transactions,missing_customer_id,missing_vendor_id
account_type,,,
Asset,315,315,315
Equity,244,244,244
Expense,4125,4125,0
Liability,202,202,202
Revenue,5514,0,5514


In [7]:
# Step 6: Check for duplicate transactions

duplicate_count = df_transactions.duplicated().sum()

print("Duplicate rows:", duplicate_count)

if duplicate_count == 0:
    print("Duplicate validation: PASSED")
else:
    print("Duplicate validation: REVIEW REQUIRED")

Duplicate rows: 0
Duplicate validation: PASSED


In [8]:
# Step 7: Validate transaction ID uniqueness

total_transaction_ids = df_transactions["transaction_id"].count()
unique_transaction_ids = df_transactions["transaction_id"].nunique()

print("Total transaction IDs:", total_transaction_ids)
print("Unique transaction IDs:", unique_transaction_ids)

if total_transaction_ids == unique_transaction_ids:
    print("Transaction ID uniqueness validation: PASSED")
else:
    print("Transaction ID uniqueness validation: REVIEW REQUIRED")

Total transaction IDs: 10400
Unique transaction IDs: 10400
Transaction ID uniqueness validation: PASSED


In [9]:
# Step 8: Inspect current data types

print("Current Data Types")
print("-" * 40)

print(df_transactions.dtypes)

Current Data Types
----------------------------------------
transaction_id       object
transaction_date     object
amount              float64
account_type         object
category             object
business_unit        object
region               object
customer_id          object
vendor_id            object
description          object
dtype: object


In [10]:
# Step 9: Validate transaction dates

parsed_dates = pd.to_datetime(
    df_transactions["transaction_date"],
    errors="coerce"
)

invalid_dates = parsed_dates.isna().sum()

print("Invalid transaction dates:", invalid_dates)

if invalid_dates == 0:
    print("Transaction date validation: PASSED")
else:
    print("Transaction date validation: REVIEW REQUIRED")

Invalid transaction dates: 0
Transaction date validation: PASSED


In [11]:
# Step 10: Convert transaction date to datetime

df_transactions["transaction_date"] = pd.to_datetime(
    df_transactions["transaction_date"]
)

print("transaction_date converted successfully!")
print("Data type:", df_transactions["transaction_date"].dtype)

transaction_date converted successfully!
Data type: datetime64[ns]


In [12]:
# Step 11: Validate transaction amounts

amount_numeric = pd.to_numeric(
    df_transactions["amount"],
    errors="coerce"
)

invalid_amounts = amount_numeric.isna().sum()

print("Invalid amounts:", invalid_amounts)

if invalid_amounts == 0:
    print("Amount validation: PASSED")
else:
    print("Amount validation: REVIEW REQUIRED")

Invalid amounts: 0
Amount validation: PASSED


In [13]:
# Step 12: Analyze transaction amount signs by account type

amount_sign_analysis = df_transactions.groupby("account_type").agg(
    total_transactions=("amount", "count"),
    positive_amounts=("amount", lambda x: (x > 0).sum()),
    negative_amounts=("amount", lambda x: (x < 0).sum()),
    zero_amounts=("amount", lambda x: (x == 0).sum()),
    minimum_amount=("amount", "min"),
    maximum_amount=("amount", "max")
)

amount_sign_analysis

,total_transactions,positive_amounts,negative_amounts,zero_amounts,minimum_amount,maximum_amount
account_type,,,,,,
Asset,315,315,0,0,3044.09,43448.79
Equity,244,244,0,0,1850.43,38423.73
Expense,4125,0,4125,0,-113310.71,-2000.00
Liability,202,202,0,0,1799.88,34373.43
Revenue,5514,5514,0,0,5000.00,351386.85


In [14]:
# Step 13: Profile categorical values

categorical_columns = [
    "account_type",
    "category",
    "business_unit",
    "region"
]

for column in categorical_columns:
    print(f"\n{column}")
    print("-" * 40)
    print("Unique values:", df_transactions[column].nunique())
    print(df_transactions[column].value_counts(dropna=False))


account_type
----------------------------------------
Unique values: 5
account_type
Revenue      5514
Expense      4125
Asset         315
Equity        244
Liability     202
Name: count, dtype: int64

category
----------------------------------------
Unique values: 12
category
Subscription    1866
Service         1851
Product         1797
Other            724
Utilities        704
Marketing        698
Supplies         688
Rent             659
Payroll          652
Asset            315
Equity           244
Liability        202
Name: count, dtype: int64

business_unit
----------------------------------------
Unique values: 3
business_unit
Online        3471
Retail        3469
Enterprise    3460
Name: count, dtype: int64

region
----------------------------------------
Unique values: 4
region
East     2673
North    2602
South    2597
West     2528
Name: count, dtype: int64


In [16]:
# Step 14: Check for blank or whitespace-only strings

string_columns = df_transactions.select_dtypes(
    include=["object"]
).columns

blank_values = {}

for column in string_columns:
    blank_count = (
        df_transactions[column]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    blank_values[column] = blank_count

blank_profile = pd.Series(
    blank_values,
    name="Blank_or_Whitespace_Count"
)

blank_profile

transaction_id       0
account_type         0
category             0
business_unit        0
region               0
customer_id       4886
vendor_id         6275
description          0
Name: Blank_or_Whitespace_Count, dtype: int64

In [17]:
 # Step 15: Create cleaned transaction dataset

df_transactions_clean = df_transactions.copy()

print("Cleaned transaction dataset created successfully!")
print("Rows:", df_transactions_clean.shape[0])
print("Columns:", df_transactions_clean.shape[1])

Cleaned transaction dataset created successfully!
Rows: 10400
Columns: 10


In [18]:
# Step 16: Validate raw vs cleaned dataset structure

print("Raw dataset shape:", df_transactions.shape)
print("Cleaned dataset shape:", df_transactions_clean.shape)

print("\nColumn structure identical:",
      df_transactions.columns.equals(df_transactions_clean.columns))

if (
    df_transactions.shape == df_transactions_clean.shape
    and df_transactions.columns.equals(df_transactions_clean.columns)
):
    print("Raw-to-cleaned structure validation: PASSED")
else:
    print("Raw-to-cleaned structure validation: REVIEW REQUIRED")

Raw dataset shape: (10400, 10)
Cleaned dataset shape: (10400, 10)

Column structure identical: True
Raw-to-cleaned structure validation: PASSED


In [19]:
# Step 17: Validate customer and vendor ID formats

customer_id_pattern = r"^CUST\d+$"
vendor_id_pattern = r"^VEND\d+$"

invalid_customer_ids = (
    df_transactions_clean["customer_id"]
    .dropna()
    .astype(str)
    .str.match(customer_id_pattern)
    .eq(False)
    .sum()
)

invalid_vendor_ids = (
    df_transactions_clean["vendor_id"]
    .dropna()
    .astype(str)
    .str.match(vendor_id_pattern)
    .eq(False)
    .sum()
)

print("Invalid customer IDs:", invalid_customer_ids)
print("Invalid vendor IDs:", invalid_vendor_ids)

if invalid_customer_ids == 0 and invalid_vendor_ids == 0:
    print("ID format validation: PASSED")
else:
    print("ID format validation: REVIEW REQUIRED")

Invalid customer IDs: 0
Invalid vendor IDs: 0
ID format validation: PASSED


In [20]:
# Step 18: Load master tables for referential validation

df_customers = pd.read_sql(
    "SELECT * FROM customers",
    conn
)

df_vendors = pd.read_sql(
    "SELECT * FROM vendors",
    conn
)

print("Customers:", df_customers.shape)
print("Vendors:", df_vendors.shape)

Customers: (400, 6)
Vendors: (120, 5)


C:\Users\Ayush Pandey\AppData\Local\Temp\ipykernel_24168\173991616.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_customers = pd.read_sql(
C:\Users\Ayush Pandey\AppData\Local\Temp\ipykernel_24168\173991616.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_vendors = pd.read_sql(


In [21]:
# Step 19: Inspect master table columns

print("Customers columns:")
print(df_customers.columns.tolist())

print("\nVendors columns:")
print(df_vendors.columns.tolist())

Customers columns:
['customer_id', 'customer_name', 'segment', 'join_date', 'region', 'status']

Vendors columns:
['vendor_id', 'vendor_name', 'category', 'region', 'active']


In [22]:
# Step 20: Validate referential integrity

customer_master_ids = set(
    df_customers["customer_id"].dropna()
)

vendor_master_ids = set(
    df_vendors["vendor_id"].dropna()
)

transaction_customer_ids = set(
    df_transactions_clean["customer_id"].dropna()
)

transaction_vendor_ids = set(
    df_transactions_clean["vendor_id"].dropna()
)

orphan_customer_ids = (
    transaction_customer_ids - customer_master_ids
)

orphan_vendor_ids = (
    transaction_vendor_ids - vendor_master_ids
)

print("Orphan customer IDs:", len(orphan_customer_ids))
print("Orphan vendor IDs:", len(orphan_vendor_ids))

if len(orphan_customer_ids) == 0 and len(orphan_vendor_ids) == 0:
    print("Referential integrity validation: PASSED")
else:
    print("Referential integrity validation: REVIEW REQUIRED")

Orphan customer IDs: 0
Orphan vendor IDs: 0
Referential integrity validation: PASSED


In [23]:
# Step 21: Validate transaction date range

min_date = df_transactions_clean["transaction_date"].min()
max_date = df_transactions_clean["transaction_date"].max()

print("Minimum transaction date:", min_date)
print("Maximum transaction date:", max_date)

if pd.notna(min_date) and pd.notna(max_date) and min_date <= max_date:
    print("Transaction date range validation: PASSED")
else:
    print("Transaction date range validation: REVIEW REQUIRED")

Minimum transaction date: 2022-01-01 00:00:00
Maximum transaction date: 2023-12-30 00:00:00
Transaction date range validation: PASSED


In [24]:
# Step 22: Validate accounting amount sign rules

sign_violations = {
    "Revenue_non_positive": (
        (df_transactions_clean["account_type"] == "Revenue") &
        (df_transactions_clean["amount"] <= 0)
    ).sum(),

    "Expense_non_negative": (
        (df_transactions_clean["account_type"] == "Expense") &
        (df_transactions_clean["amount"] >= 0)
    ).sum(),

    "Asset_non_positive": (
        (df_transactions_clean["account_type"] == "Asset") &
        (df_transactions_clean["amount"] <= 0)
    ).sum(),

    "Liability_non_positive": (
        (df_transactions_clean["account_type"] == "Liability") &
        (df_transactions_clean["amount"] <= 0)
    ).sum(),

    "Equity_non_positive": (
        (df_transactions_clean["account_type"] == "Equity") &
        (df_transactions_clean["amount"] <= 0)
    ).sum()
}

print("Accounting Sign Validation")
print("-" * 40)

for rule, count in sign_violations.items():
    print(f"{rule}: {count}")

total_violations = sum(sign_violations.values())

print("-" * 40)

if total_violations == 0:
    print("Accounting sign validation: PASSED")
else:
    print("Accounting sign validation: REVIEW REQUIRED")

Accounting Sign Validation
----------------------------------------
Revenue_non_positive: 0
Expense_non_negative: 0
Asset_non_positive: 0
Liability_non_positive: 0
Equity_non_positive: 0
----------------------------------------
Accounting sign validation: PASSED


In [25]:
# Step 23: Analyze transaction amount distribution

amount_summary = df_transactions_clean["amount"].describe()

amount_summary

count     10400.000000
mean      68456.676368
std      103041.354633
min     -113310.710000
25%      -33691.872500
50%       78624.650000
75%      157303.552500
max      351386.850000
Name: amount, dtype: float64

In [26]:
# Step 24: Calculate IQR-based outlier boundaries

Q1 = df_transactions_clean["amount"].quantile(0.25)
Q3 = df_transactions_clean["amount"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

outlier_count = (
    (df_transactions_clean["amount"] < lower_bound) |
    (df_transactions_clean["amount"] > upper_bound)
).sum()

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Potential IQR outliers:", outlier_count)

Q1: -33691.8725
Q3: 157303.55250000002
IQR: 190995.42500000002
Lower bound: -320185.01
Upper bound: 443796.69000000006
Potential IQR outliers: 0


In [27]:
# Step 25: Check for leading/trailing whitespace

text_columns = [
    "transaction_id",
    "account_type",
    "category",
    "business_unit",
    "region",
    "customer_id",
    "vendor_id",
    "description"
]

whitespace_issues = {}

for column in text_columns:
    series = df_transactions_clean[column].dropna().astype(str)

    whitespace_issues[column] = (
        series != series.str.strip()
    ).sum()

whitespace_profile = pd.Series(
    whitespace_issues,
    name="Leading_or_Trailing_Whitespace_Count"
)

whitespace_profile

transaction_id    0
account_type      0
category          0
business_unit     0
region            0
customer_id       0
vendor_id         0
description       0
Name: Leading_or_Trailing_Whitespace_Count, dtype: int64

In [28]:
# Step 26: Final cleaning validation

required_columns = [
    "transaction_id",
    "transaction_date",
    "amount",
    "account_type",
    "category",
    "business_unit",
    "region",
    "description"
]

print("Final Cleaning Validation")
print("-" * 40)

print("Rows:", len(df_transactions_clean))
print("Columns:", len(df_transactions_clean.columns))

required_nulls = df_transactions_clean[required_columns].isnull().sum().sum()
duplicate_rows = df_transactions_clean.duplicated().sum()
unique_ids = df_transactions_clean["transaction_id"].nunique()

print("Missing values in required fields:", required_nulls)
print("Duplicate rows:", duplicate_rows)
print("Unique transaction IDs:", unique_ids)

if (
    len(df_transactions_clean) == 10400
    and len(df_transactions_clean.columns) == 10
    and required_nulls == 0
    and duplicate_rows == 0
    and unique_ids == 10400
):
    print("\nFINAL CLEANING VALIDATION: PASSED")
else:
    print("\nFINAL CLEANING VALIDATION: REVIEW REQUIRED")

Final Cleaning Validation
----------------------------------------
Rows: 10400
Columns: 10
Missing values in required fields: 0
Duplicate rows: 0
Unique transaction IDs: 10400

FINAL CLEANING VALIDATION: PASSED


# Data Cleaning Summary

## Objective

The objective of this phase was to assess and prepare the financial transaction data for downstream exploratory and statistical analysis while preserving the integrity of the source data.

## Cleaning and Validation Results

| Check | Result | Action |
|---|---|---|
| Missing values | Customer and vendor IDs contain conditionally missing values | Retained as NULL because missingness is structurally consistent with account type |
| Duplicate rows | 0 | No removal required |
| Transaction ID uniqueness | 10,400 unique IDs | Passed |
| Transaction date validity | 0 invalid dates | Converted to datetime |
| Amount validity | 0 invalid values | No correction required |
| Accounting sign convention | Passed | No sign changes applied |
| Categorical values | Consistent | No standardization required |
| Blank/whitespace values | No issues | No correction required |
| Customer/Vendor ID format | Passed | No correction required |
| Referential integrity | Passed | No orphan IDs found |
| Transaction date range | 2022-01-01 to 2023-12-30 | Valid |
| IQR outlier check | 0 potential outliers | No records removed |
| Final validation | Passed | Dataset retained with 10,400 rows |

## Key Cleaning Decisions

1. `transaction_date` was converted from `object` to `datetime64[ns]`.
2. Missing `customer_id` and `vendor_id` values were retained because they follow the observed transaction/account-type structure.
3. No duplicate transactions were identified.
4. All transaction IDs are unique.
5. No invalid customer or vendor IDs were found.
6. All non-null customer and vendor IDs successfully matched their respective master tables.
7. No accounting sign violations were identified.
8. No IQR-based amount outliers were identified.
9. No unnecessary rows or financial values were removed or modified.

## Final Dataset Status

The cleaned transaction dataset contains **10,400 rows and 10 columns** and passed all final cleaning validation checks.

**Status: DATA CLEANING COMPLETED**